# 🔬 Malaria Detection using Deep Learning

This notebook trains a deep-learning model to classify microscopic blood smear
images as **Parasitized** or **Uninfected**.

## Setup Instructions
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Run all cells in order
3. The trained model will be saved as `malaria_model.h5`

## 1 — Install Dependencies

In [ ]:
!pip install -q tensorflow opencv-python scikit-learn matplotlib pillow

## 2 — Download & Extract Dataset

In [ ]:
import os, zipfile, shutil, urllib.request, sys

DATASET_URL = 'https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip'
ZIP_PATH = 'cell_images.zip'
DATASET_DIR = 'dataset'

def progress(block, block_size, total):
    pct = min(100, block * block_size * 100 / total) if total > 0 else 0
    sys.stdout.write(f'\rDownloading: {pct:.1f}%')
    sys.stdout.flush()

if not os.path.isdir(os.path.join(DATASET_DIR, 'Parasitized')):
    if not os.path.isfile(ZIP_PATH):
        print('Downloading dataset …')
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH, reporthook=progress)
        print('\nDone.')
    print('Extracting …')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall('_tmp')
    # Organise
    os.makedirs(DATASET_DIR, exist_ok=True)
    for root, dirs, _ in os.walk('_tmp'):
        if 'Parasitized' in dirs and 'Uninfected' in dirs:
            shutil.move(os.path.join(root, 'Parasitized'), os.path.join(DATASET_DIR, 'Parasitized'))
            shutil.move(os.path.join(root, 'Uninfected'), os.path.join(DATASET_DIR, 'Uninfected'))
            break
    shutil.rmtree('_tmp', ignore_errors=True)
    os.remove(ZIP_PATH) if os.path.isfile(ZIP_PATH) else None
    print('Dataset ready!')
else:
    print('Dataset already exists.')

print(f"Parasitized: {len(os.listdir(os.path.join(DATASET_DIR, 'Parasitized')))} images")
print(f"Uninfected:  {len(os.listdir(os.path.join(DATASET_DIR, 'Uninfected')))} images")

## 3 — Data Preprocessing & Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    fill_mode='nearest',
    validation_split=0.2,
)

val_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2,
)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='training', shuffle=True, seed=42
)

val_gen = val_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', shuffle=False, seed=42
)

print(f'Training samples:   {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Classes: {train_gen.class_indices}')

## 4 — Visualise Sample Images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

imgs, labels = next(train_gen)
class_names = {v: k for k, v in train_gen.class_indices.items()}

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    ax.set_title(class_names[int(labels[i])], fontsize=11)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5A — Model 1: Custom CNN

In [ ]:
from tensorflow.keras import layers, models

def build_custom_cnn():
    model = models.Sequential(name='Custom_CNN')
    # Block 1
    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(224,224,3)))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.BatchNormalization())
    # Block 2
    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.BatchNormalization())
    # Block 3
    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.BatchNormalization())
    # Head
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

cnn_model = build_custom_cnn()
cnn_model.summary()

## 5B — Model 2: MobileNetV2 (Transfer Learning)

In [ ]:
from tensorflow.keras.applications import MobileNetV2

def build_mobilenet():
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False
    model = models.Sequential(name='MobileNetV2_Transfer')
    model.add(base)
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

mobilenet_model = build_mobilenet()
mobilenet_model.summary()

## 6 — Select & Train Model

Change `USE_MODEL` to `'cnn'` to train the custom CNN instead.

In [ ]:
import tensorflow as tf
from tensorflow.keras import callbacks

USE_MODEL = 'mobilenet'   # change to 'cnn' for the custom CNN
EPOCHS = 10

model = mobilenet_model if USE_MODEL == 'mobilenet' else cnn_model

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint('malaria_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
]

print(f'GPU available: {tf.config.list_physical_devices("GPU")}')
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=cb_list)

## 7 — Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], 'o-', label='Train')
ax1.plot(history.history['val_accuracy'], 'o-', label='Val')
ax1.set_title('Accuracy vs Epochs'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], 'o-', label='Train')
ax2.plot(history.history['val_loss'], 'o-', label='Val')
ax2.set_title('Loss vs Epochs'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 8 — Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

val_gen.reset()
y_prob = model.predict(val_gen).ravel()
y_pred = (y_prob >= 0.5).astype(int)
y_true = val_gen.classes

print('Accuracy :', accuracy_score(y_true, y_pred))
print('Precision:', precision_score(y_true, y_pred))
print('Recall   :', recall_score(y_true, y_pred))
print('F1 Score :', f1_score(y_true, y_pred))
print('ROC AUC  :', roc_auc_score(y_true, y_prob))
print()
print(classification_report(y_true, y_pred, target_names=list(val_gen.class_indices.keys())))

## 9 — Confusion Matrix & ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=list(val_gen.class_indices.keys())).plot(ax=ax1, cmap='Blues')
ax1.set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
ax2.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc_score(y_true, y_prob):.4f}')
ax2.plot([0,1],[0,1],'k--', alpha=0.4)
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR'); ax2.set_title('ROC Curve')
ax2.legend()

plt.tight_layout(); plt.show()

## 10 — Single Image Prediction

In [ ]:
import cv2

def predict_single(image_path, model):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = img.astype('float32') / 255.0
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img, verbose=0)[0][0]
    if pred >= 0.5:
        return 'Uninfected', round(float(pred) * 100, 2)
    else:
        return 'Parasitized', round(float(1 - pred) * 100, 2)

# Test on a sample image
sample = os.path.join(DATASET_DIR, 'Parasitized', os.listdir(os.path.join(DATASET_DIR, 'Parasitized'))[0])
label, conf = predict_single(sample, model)
print(f'Prediction: {label}  |  Confidence: {conf}%')

plt.imshow(cv2.cvtColor(cv2.imread(sample), cv2.COLOR_BGR2RGB))
plt.title(f'{label} ({conf}%)')
plt.axis('off'); plt.show()

## 11 — Save & Download Model

In [ ]:
os.makedirs('models', exist_ok=True)
model.save('models/malaria_model.h5')
print('Model saved to models/malaria_model.h5')

# Download from Colab (uncomment if running on Colab)
# from google.colab import files
# files.download('models/malaria_model.h5')